# Step 08.3 — Time Loop Implementation：把单步 RNN 展开成完整序列

Step 8.2 已经会计算一个时间步：

$$
h_t=\tanh(W_xx_t+W_hh_{t-1}+b)
$$

现在把它真正接到 Step 7 的 sequence tensor：

```text
X.shape = (B,T,C)
```

目标是沿 T 轴逐步读取 `X[:,t,:]`，维护 hidden state，并最终收集：

```text
H_all.shape = (B,T,H)
```

本节五部分：① `X[:,t,:]` 到底取出了什么；② 完整 `for t in range(T)` 与 `mx.stack`；③ Batch 为什么可以并行而 Time 不能任意并行；④ initial/final state、不同 T 与 batching 的边界；⑤ hidden-state trajectory 可视化与 8.4 的 Module 封装准备。

注意：这里仍然用手工指定的 RNN weights，先学习时间计算结构。把 weights 正式装入 `nn.Module` 参数树放到 8.4。


## 1. 从 `(B,T,C)` 取出一个时间步：`X[:, t, :] -> (B,C)`

假设：

```text
B = 2   两条 sequence
T = 4   每条有 4 个时间步
C = 2   每个 x_t 有 2 个输入 features
H = 2   hidden state 有 2 个 features
```

整个输入 `X` 是 `(B,T,C)`。如果 t=1：

```python
x_t = X[:, 1, :]
```

含义是：

- axis 0 `:`：保留 batch 中全部样本；
- axis 1 `1`：只取第 1 个时间位置；
- axis 2 `:`：保留该 token 的全部 C 个 features。

所以 shape 从 `(B,T,C)` 变成 `(B,C)`。

这和 Step 3 的 indexing、Step 7 的 `(B,T,C)` 语义完全接上。


In [ ]:
import mlx.core as mx

B, T, C, H = 2, 4, 2, 2

X = mx.array([
    [[1.0, 0.0],
     [0.0, 1.0],
     [1.0, 1.0],
     [0.0, 0.0]],

    [[0.0, 1.0],
     [1.0, 0.0],
     [0.0, 0.0],
     [1.0, 1.0]],
])

print('X shape      :', X.shape)
for t in range(T):
    x_t = X[:, t, :]
    print(f't={t}: x_t shape={x_t.shape}, x_t={x_t}')


这里 loop 在 Python 层只跑 4 次；每一次 `x_t` 都同时包含 B 条 sequence 的当前输入。

初始化 hidden state：

```python
h = mx.zeros((B,H))
```

这表示每个 batch sample 都有自己独立的一行 hidden state。通常独立训练样本会从零状态开始；如果是明确的 stateful streaming 场景，也可以把上一 chunk 的 final state 作为下一 chunk 的 initial state，但不能把互不相关的训练样本随意串起来。


## 2. 完整时间循环：每一步更新 h，并把所有状态 stack 回 `(B,T,H)`

沿用 8.2 的单步函数：

```python
def rnn_step(x_t, h_prev, W_x, W_h, b):
    return tanh(x_t @ W_x.T + h_prev @ W_h.T + b)
```

时间循环的核心只有四步：

```text
1. 从 X 取当前 x_t
2. 用 h_prev 算新的 h
3. 保存当前 h
4. 把 h 作为下一时间步的 h_prev
```

为了和 8.2 手算结果一致，仍使用 `W_x=I`、`W_h=0.5I`、`b=0`。


In [ ]:
def rnn_step(x_t, h_prev, W_x, W_h, b):
    return mx.tanh(x_t @ W_x.T + h_prev @ W_h.T + b)

W_x = mx.array([[1.0, 0.0],
                [0.0, 1.0]])
W_h = mx.array([[0.5, 0.0],
                [0.0, 0.5]])
b = mx.array([0.0, 0.0])

h = mx.zeros((B, H))
states = []

for t in range(T):
    x_t = X[:, t, :]      # (B,C)
    h = rnn_step(x_t, h, W_x, W_h, b)  # (B,H)
    states.append(h)

H_all = mx.stack(states, axis=1)  # B copies over time -> (B,T,H)
mx.eval(H_all, h)

print('number of saved states:', len(states))
print('one saved h shape     :', states[0].shape)
print('H_all shape           :', H_all.shape)
print('final h shape         :', h.shape)
print('last saved state == final h:',
      bool(mx.allclose(H_all[:, -1, :], h)))


`states` 是长度 T 的 Python list，每个元素 shape 都是 `(B,H)`。

把它们 `mx.stack(..., axis=1)`：

```text
T 个 (B,H)
     ↓ stack at new axis 1
(B,T,H)
```

为什么选 `axis=1`？因为我们希望新的轴位于 B 和 H 之间：

```text
axis 0 = B
axis 1 = T
axis 2 = H
```

于是 `H_all[b,t,:]` 就是第 b 条 sequence 在时间 t 的 hidden state。

注意 `H_all` 保存**所有时间步**，而变量 `h` 循环结束后只保存 final state `h_(T-1)`。两者用途不同。


## 3. Batch Parallelism vs Time Dependency：B 可以一起算，T 有递推链

这一节最重要的执行直觉是：

```text
Batch axis B：同一 t 的多个样本可以一起做矩阵乘法
Time axis T ：h_t 依赖 h_(t-1)，不能随意把所有 t 当独立样本
```

在某个固定 t：

```text
x_t      : (B,C)
h_prev   : (B,H)

x_t @ W_x.T    -> (B,H)
h_prev @ W_h.T -> (B,H)
```

这两个 matmul 同时处理 B 行，所以 batch 内并行。

但是：

```text
h_0 -> h_1 -> h_2 -> h_3
```

`h_2` 需要真正的 `h_1`，因此存在数学上的时间依赖链。优化库可以融合 kernel、预先计算输入投影等，但不能简单假装这些 recurrent dependencies 不存在。


### 一个有用的分解：输入投影可以一次算完，历史更新仍然要按 t 推进

因为 `W_x` 对所有时间步相同，

```python
input_terms = X @ W_x.T
```

可以直接产生 `(B,T,H)`。

但 `h_prev @ W_h.T` 中的 `h_prev` 来自上一轮，因此仍需要 recurrence。


In [ ]:
input_terms = X @ W_x.T  # (B,T,H)，所有时间步的新输入贡献可以一起投影

h2 = mx.zeros((B, H))
states2 = []
for t in range(T):
    history_term = h2 @ W_h.T
    h2 = mx.tanh(input_terms[:, t, :] + history_term + b)
    states2.append(h2)

H_all_2 = mx.stack(states2, axis=1)
mx.eval(H_all_2)

print('input_terms shape:', input_terms.shape)
print('same result as direct loop:', bool(mx.allclose(H_all, H_all_2)))


### Batch 中的两条 sequence 会不会彼此污染？

不会，只要我们没有显式在 Batch axis 上做混合。

`x_t @ W_x.T` 和 `h_prev @ W_h.T` 对每一行使用同一套 weights，但每一行保留自己的数据与 state。这叫 **parameter sharing**，不是 state sharing。

因此：

```text
sample 0: h0 -> h1 -> h2 ...
sample 1: h0 -> h1 -> h2 ...
```

两条 recurrent chain 在逻辑上彼此独立，却可以在矩阵运算里同时执行。


## 4. Initial State、Final State 与“可变长度”要准确理解

RNN 的参数不依赖 T，因此同一个 `W_x,W_h,b` 可以处理 T=2、T=4、T=20……这就是“结构上支持不同 sequence length”的重要含义。

但 tensor batch 仍必须是规则矩形。例如一个 batch 中真实长度分别为 3、5、7 时，不能直接形成普通 `(B,T,C)` 而不给短序列补齐。

实际系统通常会使用 padding + length/mask、packing，或其它 batch 组织方式。**RNN 支持变长序列，不等于一个普通 dense tensor 可以自动容纳每行不同 T。**

本课程核心先使用统一 T 的小 batch，等数据管线阶段再深入 padding/packing。

另一个区别：

```text
H_all     = (B,T,H)  所有时刻的状态
h_final   = (B,H)    每条 sequence 的最后状态
```

历史上，很多 sequence classification / encoder 架构会特别使用 final state；sequence-to-sequence 或逐步输出任务则往往需要所有状态。后面 Seq2Seq/Attention 会再次看到这个区别。


In [ ]:
def run_rnn(X_seq, W_x, W_h, b):
    B_local, T_local, _ = X_seq.shape
    H_local = W_h.shape[0]
    h_local = mx.zeros((B_local, H_local))
    saved = []

    for t in range(T_local):
        h_local = rnn_step(X_seq[:, t, :], h_local, W_x, W_h, b)
        saved.append(h_local)

    return mx.stack(saved, axis=1), h_local

H_short, final_short = run_rnn(X[:, :2, :], W_x, W_h, b)
H_full, final_full = run_rnn(X, W_x, W_h, b)
mx.eval(H_short, H_full, final_short, final_full)

print('same weights, T=2 ->', H_short.shape, final_short.shape)
print('same weights, T=4 ->', H_full.shape, final_full.shape)


这里 `run_rnn` 已经足够展示完整时间循环，但还只是普通 Python 函数，权重作为显式参数传入。

下一节 8.4 会把 `W_x,W_h,b` 正式装进一个 `nn.Module`，让 MLX 自动把它们放进 parameter tree。


## 5. Hidden-State Trajectory：把 `(T,H)` 看成一条随时间演化的内部轨迹

固定一个 batch sample：

```text
H_all[b].shape = (T,H)
```

每一行是一个时间步，每一列是一个 hidden feature。可以画 heatmap，看状态如何随新输入逐步变化。

这里要谨慎：hidden feature 0、1 并不自动等于某种人类可命名语义；这幅图只是显示内部数值轨迹。


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, B, figsize=(9, 4), sharey=True)

for batch_index, ax in enumerate(axes):
    matrix = H_all[batch_index].tolist()
    image = ax.imshow(matrix, aspect='auto', vmin=-1, vmax=1)
    ax.set_title(f'batch sample {batch_index}')
    ax.set_xlabel('hidden feature h')
    ax.set_xticks(range(H))
    ax.set_yticks(range(T))

axes[0].set_ylabel('time step t')
fig.colorbar(image, ax=axes.ravel().tolist(), label='hidden-state value')
plt.show()


绘图技巧：并排比较多个矩阵时使用相同 `vmin/vmax`，否则每张 heatmap 自动缩放会让数值大小难以公平比较；`sharey=True` 让两个 sample 共用时间轴。

### 8.3 最终结论

- `(B,T,C)` 中 `X[:,t,:]` 会取出当前时间步全部 batch samples，shape 为 `(B,C)`。
- 初始 hidden state 通常设为 `(B,H)` 的零矩阵；每个 batch sample 拥有自己的 state。
- `for t in range(T)` 按时间更新 `h_t`，而 `W_x,W_h,b` 在所有 t 共享。
- 保存 T 个 `(B,H)` 状态并沿 axis 1 `stack`，得到 `(B,T,H)`。
- Batch 轴可以通过矩阵运算并行；同一 sequence 的 Time 轴存在 `h_(t-1) -> h_t` 的真实递推依赖。
- 输入投影 `X @ W_x.T` 可以对所有 t 一次计算，但 recurrent history path 仍依赖前一状态。
- RNN 参数形状不依赖 T，因此同一 cell 可处理不同长度；但把不同真实长度组成同一个 dense batch 仍需 padding/masking/packing 等工程处理。
- `H_all` 与 final hidden state 用途不同，后面 Seq2Seq/Attention 会重新出现。
- 下一节 8.4：把这些逻辑封装成一个真正的 Apple MLX `nn.Module` RNN Cell，检查 parameter tree、shape 和跨时间共享参数。
